In [ ]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

from qd_solve import *
from qd_solve.eig import *
from qd_solve.operator import *
from qd_solve.solver import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

In [ ]:
x0 = -10
xf = 10
num_steps = 1000
mesh = Mesh(x0, xf, num_steps)

potential = lambda x: 0.5 * x ** 2
V = PotentialEnergy(potential)
T = KineticEnergy()
H = T + V

In [ ]:
%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh(H, mesh, 1000))
%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh(H, mesh, 1000))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

idx = 15
plt.plot(mesh.x_range, jnp.real(y_eigvecs[idx].values))
plt.plot(mesh.x_range, jnp.imag(y_eigvecs[idx].values))

plt.savefig("dense_eig15.png")
plt.show()
plt.close(fig)

In [ ]:
# seems to only work when num_iterations is close to num_modes
%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh_lanczos(\
    H, mesh, num_modes=1000, num_eig=16, num_iterations=800))

%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh_lanczos(\
    H, mesh, num_modes=1000, num_eig=16, num_iterations=800))

residuals

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

idx = 15
plt.plot(mesh.x_range, jnp.real(y_eigvecs[idx].values))
plt.plot(mesh.x_range, jnp.imag(y_eigvecs[idx].values))

plt.savefig("lanczos_eig15.png")
plt.show()
plt.close(fig)

In [ ]:
%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh_restart(\
    H, mesh, num_modes=1000, num_eig=16, num_iterations=100, buffer=50, num_restarts=25))

%time eigvals, y_eigvecs, residuals = jax.block_until_ready(op_eigh_restart(\
    H, mesh, num_modes=1000, num_eig=16, num_iterations=100, buffer=50, num_restarts=25))

residuals

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)

idx = 15
plt.plot(mesh.x_range, jnp.real(y_eigvecs[idx].values))
plt.plot(mesh.x_range, jnp.imag(y_eigvecs[idx].values))

plt.savefig("restart_eig15.png")
plt.show()
plt.close(fig)